# 🧬 Обучение «второй копии» Persona на Colab (бесплатно)

Бесплатный способ обучить твою личную модель: Colab даёт GPU **T4 (16 ГБ)** даром.
Обучаем QLoRA на твоём датасете → конвертируем в **GGUF** → скачиваешь → запускаешь на ПК (1050 Ti) через Ollama.

**Шаг 0 (обязательно):** меню → *Runtime → Change runtime type → Hardware accelerator = **T4 GPU*** → Save.
Потом *Runtime → Run all* (или ячейки по очереди). Весь прогон ~20–40 мин.


In [ ]:
# Проверка GPU (должна показать Tesla T4)
!nvidia-smi -L || echo 'GPU нет — включи T4 в Runtime → Change runtime type'


## 1. Поставить ML-библиотеки (~2 мин)


In [ ]:
%pip -q install -U "transformers>=4.44" "peft>=0.12" "trl>=0.10" "datasets>=2.20" "bitsandbytes>=0.43" "accelerate>=0.33" sentencepiece


## 2. Загрузить датасет
Нажми кнопку и выбери **`persona.jsonl`** (и если есть — **`persona.val.jsonl`**).

> Файлы на сервере (RDP): `C:\www-Yaroslav\Persona\finetune\data\`. Если открыл Colab в RDP-браузере — бери оттуда. Либо скачай их со страницы `/settings/integrations` в Persona.


In [ ]:
from google.colab import files
up = files.upload()   # выбери persona.jsonl (+ persona.val.jsonl)
print('Загружено:', list(up.keys()))


## 3. Обучение (QLoRA)
База по умолчанию **Qwen2.5-1.5B** (на T4 заходит, качество лучше). Полегче → `Qwen/Qwen2.5-0.5B-Instruct`.
Это клон СТИЛЯ и самосознания (имя Persona, «ты мой автор»), не интеллекта.


In [ ]:
import json, os, torch, inspect
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL   = 'Qwen/Qwen2.5-1.5B-Instruct'   # полегче: 'Qwen/Qwen2.5-0.5B-Instruct'
DATA    = 'persona.jsonl'
VAL     = 'persona.val.jsonl'            # нет файла — поставь ''
OUT     = 'persona-lora'
EPOCHS  = 3.0
MAXSEQ  = 1024

def read_jsonl(p):
    if not p or not os.path.exists(p): return []
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

def fmt(rows):
    txt = [tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False) for r in rows]
    return Dataset.from_dict({'text': txt})

train_ds = fmt(read_jsonl(DATA))
val_rows = read_jsonl(VAL)
eval_ds  = fmt(val_rows) if val_rows else None
assert len(train_ds) > 0, 'persona.jsonl пуст или не загружен — вернись к шагу 2'

# TRL меняет имена аргументов между версиями (max_seq_length -> max_length,
# dataset_text_field иногда убирают) — собираем kwargs по реальной сигнатуре.
sft_kwargs = dict(output_dir=OUT, num_train_epochs=EPOCHS, per_device_train_batch_size=1,
                  gradient_accumulation_steps=16, learning_rate=2e-4, fp16=True, bf16=False,
                  gradient_checkpointing=True, logging_steps=10, save_strategy='epoch',
                  optim='paged_adamw_8bit', warmup_ratio=0.03, lr_scheduler_type='cosine',
                  report_to='none')
params = inspect.signature(SFTConfig.__init__).parameters
if 'max_seq_length' in params: sft_kwargs['max_seq_length'] = MAXSEQ
elif 'max_length' in params:   sft_kwargs['max_length'] = MAXSEQ
if 'dataset_text_field' in params: sft_kwargs['dataset_text_field'] = 'text'
cfg = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds, peft_config=lora)
print(f'Обучаю {MODEL} на {len(train_ds)} примерах…')
trainer.train()
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('LoRA-адаптер готов ->', OUT)


## 4. Слить LoRA в базовую модель


In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
m = AutoPeftModelForCausalLM.from_pretrained('persona-lora')
m = m.merge_and_unload()
m.save_pretrained('persona-merged')
AutoTokenizer.from_pretrained('persona-lora').save_pretrained('persona-merged')
print('merged -> persona-merged')


## 5. Конвертация в GGUF + квант Q4_K_M
Q4_K_M комфортно влезает в 4 ГБ 1050 Ti. Сборка llama.cpp ~3–5 мин.


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py persona-merged --outfile persona-f16.gguf
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF >/dev/null
!cmake --build llama.cpp/build -j --target llama-quantize >/dev/null
!./llama.cpp/build/bin/llama-quantize persona-f16.gguf persona-q4.gguf Q4_K_M
import os; print('GGUF:', [f for f in os.listdir('.') if f.endswith('.gguf')])


## 6. Скачать готовую модель
Скачается `persona-q4.gguf` (~1 ГБ для 1.5B). Положи на свой ПК.


In [ ]:
from google.colab import files
files.download('persona-q4.gguf')


## 7. На твоём ПК (1050 Ti) — запуск через Ollama
Рядом с `persona-q4.gguf` создай файл **`Modelfile`**:
```
FROM ./persona-q4.gguf
PARAMETER temperature 0.7
PARAMETER num_ctx 4096
SYSTEM "Ты — Persona, личный ИИ-друг этого человека. Тёплый, прямой, на «ты», на его стороне, по-русски."
```
Затем:
```
ollama create persona-mini -f Modelfile
ollama run persona-mini
```
В Persona → **/settings/llm** → провайдер `ollama`, модель `persona-mini`.

---
**Если ругнулось:**
- `SFTConfig got unexpected keyword` — обычно версия TRL; ячейка обучения уже собирает аргументы по сигнатуре, так что должно пройти. Если всё же — `%pip install -U trl` и заново.
- OOM → `MODEL='Qwen/Qwen2.5-0.5B-Instruct'`, `MAXSEQ=512`, *Runtime → Restart*, заново.
- нет `convert_hf_to_gguf.py` → `!ls llama.cpp/*.py` (путь мог измениться).
